In [ ]:
import sys
from pathlib import Path

_SRC = Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fraud_detect import config, io, viz, models, evaluation as ev, tuning
from fraud_detect.models import ModelBackend

warnings.filterwarnings("ignore")
viz.configure_style()

## 15. Final Summary

### Pipeline

```
01 ──► 02 ──► 03 ──► 04 ──► 05 ──► 06 ──► 07 ──► 08 ──► 09 ──► 10 ──► 11 ──► 12 ──► 13 ──► 14 ──► 15
```

| # | Notebook | Description |
|---|---|---|
| 01 | Data Loading | Merge transaction + identity |
| 02 | EDA Transaction | Amount, product codes, card features |
| 03 | EDA Identity | Device info, browser, OS |
| 04 | Missing Values | Imputation strategies |
| 05 | Target Imbalance | ~3.5% fraud |
| 06 | Correlation | 132 redundant features |
| 07 | Feature Engineering | Time, amount, email, card |
| 08 | Feature Importance | LightGBM selection |
| 09 | Baseline | Logistic regression |
| 10 | Advanced Models | LightGBM, XGBoost, CatBoost |
| 11 | Tuning | Optuna optimisation |
| 12 | Ensemble | Voting + stacking |
| 13 | Evaluation | ROC, PR, McNemar |
| 14 | Error Analysis | Segmentation, shift, FP/FN |
| 15 | Final Summary | This notebook |

### Data Overview
- ~590K transactions, 400+ features, 3.5% fraud rate

In [ ]:
train = io.load_train_features()
print(f"Training data shape: {train.shape}")
print(f"Fraud rate: {train['isFraud'].mean()*100:.2f}%")

### 15.1 Final Model Performance

In [ ]:
# Train all tuned models
final_results = {}
for backend in ModelBackend:
    params = tuning.load_best_params(backend, fallback_to_defaults=True)
    result = models.train_model(train, backend=backend, params=params)
    final_results[backend.value] = result
    print(f"{backend.value:12s} | Val AUC: {result.val_auc:.5f} | Time: {result.training_time:.1f}s")

In [ ]:
split = models.make_train_val_split(train)

reports = {}
for name, result in final_results.items():
    preds = result.model.predict_proba(split.X_val)[:, 1]
    reports[name] = ev.compute_evaluation(split.y_val, preds, model_name=name)

ev.compare_models(reports)

### 15.2 Best Model

In [ ]:
best_name = comparison.iloc[0]["model"]
best_report = reports[best_name]
print(f"Best model: {best_name.upper()}")
print(f"  ROC-AUC   : {best_report.auc:.5f}")
print(f"  PR-AUC    : {best_report.average_precision:.5f}")
print(f"  F1        : {best_report.f1:.5f}")
print(f"  Precision : {best_report.precision:.5f}")
print(f"  Recall    : {best_report.recall:.5f}")
print(f"  Threshold : {best_report.best_threshold:.4f}")

In [ ]:
best_preds = final_results[best_name].model.predict_proba(split.X_val)[:, 1]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ROC curve
viz.plot_roc_curves({best_name: split.y_val}, {best_name: best_preds}, ax=axes[0, 0])

# PR curve
viz.plot_pr_curves({best_name: split.y_val}, {best_name: best_preds}, ax=axes[0, 1])

# Confusion matrix
viz.plot_confusion_matrix(
    split.y_val,
    (best_preds >= best_report.best_threshold).astype(int),
    ax=axes[1, 0],
)

# Cumulative gain
viz.plot_cumulative_gain(split.y_val, best_preds, ax=axes[1, 1])

plt.tight_layout()
viz.save_figure(fig, config.METADATA_DIR / "best_model_dashboard.png")
plt.show()

### 15.3 Reproduce

```bash
pip install -e ".[lgbm,xgb,cat,dev]"
python data_prep.py          # CSV -> Parquet
jupyter notebook notebook/    # run 01 -> 15
make test                     # 94 tests
```

### 15.4 Results

| Target | AUC | Status |
|---|---|---|
| Bronze (>0.90) | | ✅ |
| Silver (>0.93) | | |
| Gold (>0.95) | | |

### Achievements
- 11 clean, typed, tested Python modules
- 3 gradient-boosting models + Optuna tuning
- Voting and stacking ensembles
- Comprehensive evaluation + error analysis
- Sphinx docs, CI/CD, 94 tests

### Tech Stack
Python 3.10+ | Pandas | NumPy | LightGBM | XGBoost | CatBoost | Optuna |
Scikit-learn | Matplotlib | Seaborn | Pytest | Hypothesis | Ruff | Sphinx